# Hugging Face Integration with AdvSecureNet API

This notebook demonstrates how to use Hugging Face datasets and models with AdvSecureNet through the Python API. We'll cover:

1. **Dataset Configuration with Splits**: How to load different splits and configure custom split mappings
2. **Hugging Face Models**: Using pretrained models from Hugging Face Hub
3. **Adversarial Attack Example**: FGSM attack using Hugging Face components
4. **Adversarial Training Example**: Training with adversarial examples

## Key Features Demonstrated
- Split priority system (`split_config` vs `load_splits` vs defaults)
- Custom split names and mappings
- Hugging Face dataset integration
- Model loading from Hugging Face Hub

## 1. Setup and Imports

## Important: Hugging Face Identifier Formats

AdvSecureNet supports multiple formats for Hugging Face identifiers:

### For Datasets:
- ✅ **Short form**: `"uoft-cs/cifar10"`
- ✅ **Full URL**: `"https://huggingface.co/datasets/uoft-cs/cifar10"`
- ✅ **Domain variations**: `"huggingface.co/datasets/uoft-cs/cifar10"`, `"hf.co/datasets/uoft-cs/cifar10"`

### For Models:
- ✅ **Short form**: `"microsoft/resnet-18"`
- ✅ **Full URL**: `"https://huggingface.co/microsoft/resnet-18"`
- ✅ **Domain variations**: `"huggingface.co/microsoft/resnet-18"`, `"hf.co/microsoft/resnet-18"`

**Key Point**: All formats work identically - use whichever is most convenient for your workflow!

In [22]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# AdvSecureNet imports
from advsecurenet.dataloader.data_loader_factory import DataLoaderFactory
from advsecurenet.models.model_factory import ModelFactory
from advsecurenet.datasets.dataset_factory import DatasetFactory
from advsecurenet.computer_vision.image_classification.attacks.gradient_based import FGSM
from advsecurenet.computer_vision.image_classification.attacks.attacker import Attacker
from advsecurenet.trainer.trainer import Trainer

# Configuration imports for API usage (not CLI configs)
from advsecurenet.shared.types.configs.model_config import CreateModelConfig
from advsecurenet.shared.types.configs.attack_configs import FgsmAttackConfig
from advsecurenet.shared.types.configs.attack_configs.attacker_config import AttackerConfig
from advsecurenet.shared.types.configs import TrainConfig
from advsecurenet.shared.types.configs.preprocess_config import PreprocessConfig, PreprocessStep
from advsecurenet.shared.types.configs.device_config import DeviceConfig

## 2. Dataset Configuration Examples

### Understanding API vs CLI Patterns

**API Usage**: Use direct parameters with `DatasetFactory.load_dataset(**kwargs)`

The dataset loading follows a priority system:
1. **Highest Priority**: `split_config` - Dictionary mapping of logical split names to split configurations
2. **Medium Priority**: `load_splits` - List of split names to load
3. **Default**: `['train', 'test']` if nothing is specified

### Example 1: Using default behaviour

In [23]:
# Example 1: Default behavior - when you don't specify load_splits or split_config
# The factory will automatically load ['train', 'test'] splits

# Define preprocessing configuration once and reuse
preprocessing_config = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 32}),
        PreprocessStep(name="CenterCrop", params={"size": 32}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(name="ToDtype", params={"dtype": "torch.float32", "scale": True}),
        PreprocessStep(
            name="Normalize",
            params={"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]}
        ),
    ]
)

print("=== Example 1: Default Behavior ===")
print("When you don't specify load_splits or split_config, the factory defaults to ['train', 'test']")

# Load dataset with minimal parameters - uses defaults
default_dataset = DatasetFactory.load_dataset(
    dataset_name="uoft-cs/cifar10",
    num_classes=10,
    preprocessing=preprocessing_config,
    constructor_args={
        "input_key": "img",
        "target_key": "label"
    }
)

print(f"Dataset loaded with default splits: {list(default_dataset.keys())}")
print(f"Train samples: {len(default_dataset['train'])}")
print(f"Test samples: {len(default_dataset['test'])}")

=== Example 1: Default Behavior ===
When you don't specify load_splits or split_config, the factory defaults to ['train', 'test']


/Users/fabienmorgan/Desktop/Ausbildung/Master/Masters_Project/Code/venv_master_project/lib/python3.13/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Dataset loaded with default splits: ['train', 'test']
Train samples: 50000
Test samples: 10000


### Example 2: Using load_splits with Custom Split Names

When you want to explicitly specify which splits to load or use custom split names:

In [24]:
# Example 2: Using load_splits to explicitly specify which splits to load
# This is useful when you want different split names or to load specific splits

print("=== Example 2: Explicit load_splits ===")
print("Explicitly specify which splits to load and their logical names")

# Load dataset with explicit split specification
explicit_splits_dataset = DatasetFactory.load_dataset(
    dataset_name="huggingface",
    identifier="uoft-cs/cifar10",
    num_classes=10,
    preprocessing=preprocessing_config,
    constructor_args={
        "input_key": "img",
        "target_key": "label"
    },
    load_splits=["train", "test"]  # Explicitly specify splits to load
)

print(f"Dataset loaded with explicit splits: {list(explicit_splits_dataset.keys())}")
print(f"Train samples: {len(explicit_splits_dataset['train'])}")
print(f"Test samples: {len(explicit_splits_dataset['test'])}")

=== Example 2: Explicit load_splits ===
Explicitly specify which splits to load and their logical names
Dataset loaded with explicit splits: ['train', 'test']
Train samples: 50000
Test samples: 10000


### Example 3: Using split_config for Different Settings Per Split

When you need different datasets, preprocessing, or split names for different logical splits:

In [25]:
# Example 3: Using split_config for different datasets with semantic compatibility
# This demonstrates the most powerful approach - different datasets with SAME semantic classes

print("=== Example 3: split_config with Semantically Compatible Datasets ===")
print("Use split_config when you need different datasets with SAME semantic meanings")

# Define different preprocessing for corrupted test set
corrupted_preprocessing = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 32}),
        PreprocessStep(name="CenterCrop", params={"size": 32}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(name="ToDtype", params={"dtype": "torch.float32", "scale": True}),
        PreprocessStep(
            name="Normalize",
            params={"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]}  # Different normalization for corrupted images
        ),
    ]
)

# Load dataset with split_config - different datasets with SAME semantic classes
split_config_dataset = DatasetFactory.load_dataset(
    dataset_name="huggingface",
    num_classes=10,
    split_config={
        "train": {
            # Train on clean CIFAR-10 images - explicit settings
            "identifier": "uoft-cs/cifar10",
            "split_name": "train",
            "preprocessing": preprocessing_config,  # Clean preprocessing for training
            "constructor_args": {  # Explicit constructor args for CIFAR-10
                "input_key": "img",
                "target_key": "label"
            }
        },
        "test": {
            # Test on corrupted CIFAR-10-C images - SAME semantic classes!
            "identifier": "randall-lab/cifar10-c",  # Different dataset but same classes
            "split_name": "test",
            "preprocessing": corrupted_preprocessing,  # Different preprocessing for corrupted images
            "constructor_args": {  # Different field names for CIFAR-10-C
                "input_key": "image",   # CIFAR-10-C uses "image" instead of "img"
                "target_key": "label"   # Same label field
            },
            "dataset_kwargs": {"trust_remote_code": True, "cache_dir": "cifar10-c/"}  # Required for CIFAR-10-C dataset
        }
    }
)

print(f"Dataset loaded with split_config: {list(split_config_dataset.keys())}")
print(f"Train samples (CIFAR-10): {len(split_config_dataset['train'])}")
print(f"Test samples (CIFAR-10-C): {len(split_config_dataset['test'])}")

=== Example 3: split_config with Semantically Compatible Datasets ===
Use split_config when you need different datasets with SAME semantic meanings
Dataset loaded with split_config: ['train', 'test']
Train samples (CIFAR-10): 50000
Test samples (CIFAR-10-C): 950000


### Example 4: Anti-Pattern - Don't Do This! (Demonstration Only)

**⚠️ WARNING: This example shows what NOT to do!**  
This demonstrates how `split_config` overrides `load_splits`, making the `load_splits` parameter pointless. This is confusing and should be avoided in real code.

In [26]:
# Example 4: Anti-Pattern - DON'T DO THIS! 
# This shows multiple confusing patterns that should be avoided

print("=== Example 4: Anti-Pattern - What NOT to Do ===")
print("❌ This example shows confusing configuration that should be avoided!")
print("⚠️  Multiple anti-patterns combined to show what makes code confusing")

# Define preprocessing that will be ignored
ignored_preprocessing = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 64}),  # This will be completely ignored!
        PreprocessStep(name="ToTensor"),
    ]
)

# BAD: Multiple confusing patterns in one configuration
antipattern_dataset = DatasetFactory.load_dataset(
    dataset_name="huggingface",
    identifier="uoft-cs/cifar10",
    num_classes=10,
    preprocessing=ignored_preprocessing,  # ❌ IGNORED: Both splits override this global preprocessing!
    constructor_args={  # ❌ PARTIALLY IGNORED: Only custom_test uses global args
        "input_key": "img",      # ❌ IGNORED: custom_train overrides this
        "target_key": "label"    # ✅ Used by custom_test (should be explicit for clarity)
    },
    load_splits=["validation", "non_existent"],  # ❌ COMPLETELY IGNORED: split_config takes priority
    split_config={  # ✅ This takes priority and OVERRIDES everything above
        "custom_train": {
            "split_name": "train",
            "preprocessing": preprocessing_config,  # Overrides global preprocessing
            "constructor_args": {  # Overrides global constructor_args
                "input_key": "img",     # Same value but redundant override
                "target_key": "label"   # Same value but redundant override  
            }
        },
        "custom_test": {
            "split_name": "test",
            "preprocessing": preprocessing_config,  # Overrides global preprocessing
            # ❌ UNCLEAR: Uses global constructor_args - should be explicit for clarity!
            # Should add: "constructor_args": {"input_key": "img", "target_key": "label"}
        }
    }
)

print(f"❌ Specified load_splits: ['train', 'test'] - COMPLETELY IGNORED!")
print(f"❌ Global preprocessing (size=64) - IGNORED by ALL splits!")
print(f"❌ Global input_key override - IGNORED by custom_train!")
print(f"❌ Global constructor_args - UNCLEAR which splits use them!")
print(f"✅ Actual splits loaded: {list(antipattern_dataset.keys())}")

print("\n❌ What Makes This Confusing:")
print("  1. load_splits specified but completely ignored")
print("  2. Global preprocessing ignored by ALL splits - wasteful!")
print("  3. Global constructor_args partially ignored - unclear!")
print("  4. Inconsistent split configurations - some explicit, some implicit")
print("  5. Redundant overrides with same values")

print("\n✅ How to Fix This:")
print("  1. Remove load_splits - pick ONE approach!")
print("  2. Remove global preprocessing if all splits override it")
print("  3. Be explicit about constructor_args in ALL splits")
print("  4. Use global settings only when actually shared")
print("  5. Don't override with identical values")

=== Example 4: Anti-Pattern - What NOT to Do ===
❌ This example shows confusing configuration that should be avoided!
⚠️  Multiple anti-patterns combined to show what makes code confusing
❌ Specified load_splits: ['train', 'test'] - COMPLETELY IGNORED!
❌ Global preprocessing (size=64) - IGNORED by ALL splits!
❌ Global input_key override - IGNORED by custom_train!
❌ Global constructor_args - UNCLEAR which splits use them!
✅ Actual splits loaded: ['custom_train', 'custom_test']

❌ What Makes This Confusing:
  1. load_splits specified but completely ignored
  2. Global preprocessing ignored by ALL splits - wasteful!
  3. Global constructor_args partially ignored - unclear!
  4. Inconsistent split configurations - some explicit, some implicit
  5. Redundant overrides with same values

✅ How to Fix This:
  1. Remove load_splits - pick ONE approach!
  2. Remove global preprocessing if all splits override it
  3. Be explicit about constructor_args in ALL splits
  4. Use global settings only w

## Summary and Best Practices

### Configuration Priority System
1. **`split_config`** (highest priority) - Use for complex split configurations with different preprocessing per split
2. **`load_splits`** (medium priority) - Use for simple split selection from available dataset splits  
3. **Default behavior** (lowest priority) - Loads `['train', 'test']` automatically

### When to Use Each Approach

**Use Default (Example 1):** When you want standard train/test splits with same preprocessing
```python
# Simple case - just specify the essentials
dataset = DatasetFactory.load_dataset(
    dataset_name="huggingface",
    identifier="uoft-cs/cifar10",
    num_classes=10,
    preprocessing=preprocessing_config,
    constructor_args={"input_key": "img", "target_key": "label"}
)
```

**Use load_splits (Example 2):** When you need specific splits but same preprocessing for all
```python
# Need specific splits, same preprocessing
dataset = DatasetFactory.load_dataset(
    load_splits=["train", "validation", "test"]  # Different splits, same preprocessing
)
```

**Use split_config (Example 3):** When you need different preprocessing or settings per split
```python
# Different preprocessing per split
split_config={
    "train": {"split_name": "train", "preprocessing": train_preprocessing},
    "test": {"split_name": "test", "preprocessing": test_preprocessing}
}
```

### ❌ Anti-Patterns to Avoid
- **Don't mix approaches** - Using both `load_splits` and `split_config` is confusing
- **Don't set global attributes that are overwritten in split_config**

### Understanding Parameter Inheritance and Override

The `DatasetFactory.load_dataset()` supports a flexible configuration system where you can set parameters globally (applied to all splits) or per-split (overriding global settings). Here's a comprehensive guide to which parameters can be set where:

### 🌍 Global Parameters (Apply to All Splits)
Set these at the top level of `DatasetFactory.load_dataset()`:

| Parameter | Description | When to Use Globally |
|-----------|-------------|---------------------|
| `dataset_name` | Dataset type ("huggingface", "torchvision", etc.) | ✅ **Always** - Same dataset system for all splits |
| `identifier` | HuggingFace dataset ID | ✅ When all splits use same dataset |
| `num_classes` | Number of output classes | ✅ **Always** - Model compatibility requires same classes |
| `preprocessing` | Preprocessing pipeline | ✅ When all splits use same preprocessing |
| `constructor_args` | Field mappings (input_key, target_key) | ✅ When all splits have same field names |
| `load_splits` | List of splits to load | ✅ Simple cases with uniform settings |
| `dataset_kwargs` | Additional dataset parameters | ✅ When all splits need same extra parameters |

### 🎯 Split-Specific Parameters (Override Global Settings)
Set these inside `split_config` for individual splits:

| Parameter | Description | When to Override Per Split |
|-----------|-------------|---------------------------|
| `identifier` | Different dataset for this split | ✅ Cross-dataset evaluation (CIFAR-10 → CIFAR-10-C) |
| `split_name` | Which split to load from dataset | ✅ **Always** in split_config - Required |
| `preprocessing` | Different preprocessing pipeline | ✅ Different augmentations/normalization per split |
| `constructor_args` | Different field mappings | ✅ Datasets with different field names |
| `dataset_kwargs` | Split-specific parameters | ✅ trust_remote_code, cache_dir per dataset |

## 3. Hugging Face Model Configuration

In [27]:
# Configure a Hugging Face model using proper attributes
model_config = CreateModelConfig(
    model_name="huggingface",  # This tells the factory it's a HF model
    model_identifier="microsoft/resnet-18",  # Use model_identifier for HF model ID
    pretrained=True,  # Direct attribute, not in model_kwargs
    trust_remote_code=False,  # Direct attribute for HF models
    architecture={"num_classes": 10},  # Model architecture parameters
    # Optional HF-specific parameters
    revision=None,  # Use specific model revision if needed
    cache_dir=None,  # Use default cache directory
)

# Load the model using the ModelFactory
model = ModelFactory.create_model(config=model_config)
print(f"Model loaded: {type(model).__name__}")
print(f"Model device: {next(model.parameters()).device}")

# Move to appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
print(f"Model moved to: {device}")
print(f"\n=== Model Configuration ===")
print(f"Model name: {model_config.model_name}")
print(f"Model identifier: {model_config.model_identifier}")
print(f"Pretrained: {model_config.pretrained}")
print(f"Architecture: {model_config.architecture}")

/Users/fabienmorgan/Desktop/Ausbildung/Master/Masters_Project/Code/advsecurenet_mp/advsecurenet/models/huggingface_model.py:189: UserWarning: Architecture arguments are applied via config for non-pretrained models. Ignoring for pretrained loading.
  warnings.warn(


Model loaded: HuggingFaceModel
Model device: cpu
Model moved to: mps

=== Model Configuration ===
Model name: huggingface
Model identifier: microsoft/resnet-18
Pretrained: True
Architecture: {'num_classes': 10}


## 4. Adversarial Attack Example

### FGSM Attack with Hugging Face Components

**Note**: FGSM attacks only need a single dataset to generate adversarial examples from. We'll use our test dataset to demonstrate the attack, but you could use any dataset with appropriate preprocessing.

In [28]:
# Configure FGSM attack using our loaded model
# Following the same pattern as adversarial_attacks.ipynb

# Setup device configuration for AdvSecureNet
device_config = DeviceConfig(processor=str(device))

# Configure FGSM attack
attack_config = FgsmAttackConfig(
    epsilon=0.03,
    targeted=False,
    device=device_config,  # Use DeviceConfig as in adversarial_attacks.ipynb
)

# Create attack instance
fgsm_attack = FGSM(config=attack_config)
print(f"Attack created: {type(fgsm_attack).__name__}")

# Use our already loaded model and test dataset for the attack
print(f"Using model: {type(model).__name__}")
print(f"Available datasets: {list(split_config_dataset.keys())}")

# Create data loader from test dataset for attack demonstration
# Note: We only need one dataset for FGSM attack, using test set here
attack_loader = DataLoaderFactory.create_dataloader(
    dataset=split_config_dataset['test'], 
    batch_size=32,
    shuffle=False
)

print(f"Created data loader for attack!")
print(f"Attack dataset: {len(attack_loader.dataset)} samples")
print(f"Device config: {device_config.processor}")

# Prepare model and get a sample batch for demonstration
model.eval()

# Get a batch of data for demonstration purposes
data_iter = iter(attack_loader)
images, labels = next(data_iter)
images, labels = images.to(device), labels.to(device)

print(f"\n=== Attack Configuration ===")
print(f"Batch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Epsilon: {attack_config.epsilon}")
print(f"Attack type: {'Targeted' if attack_config.targeted else 'Untargeted'}")
print(f"Device: {device_config.processor}")

Attack created: FGSM
Using model: HuggingFaceModel
Available datasets: ['train', 'test']
Created data loader for attack!
Attack dataset: 950000 samples
Device config: mps

=== Attack Configuration ===
Batch shape: torch.Size([32, 3, 32, 32])
Labels shape: torch.Size([32])
Epsilon: 0.03
Attack type: Untargeted
Device: mps


In [29]:
# Execute the attack using the Attacker class (recommended approach)
# This follows the same pattern as in adversarial_attacks.ipynb

# Configure the attacker
attacker_config = AttackerConfig(
    model=model,
    attack=fgsm_attack,
    dataloader=attack_loader,
    device=DeviceConfig(processor=str(device)),
    return_adversarial_images=True,
)

# Create the attacker instance
attacker = Attacker(config=attacker_config)

# Execute the attack - this returns all adversarial images
adversarial_images_all = attacker.execute()

print(f"Attack completed!")
print(f"Generated adversarial images: {len(adversarial_images_all)} batches")

# For demonstration, let's also show the manual approach for one batch
print(f"\n🔄 Demonstrating manual attack on one batch for comparison...")

model.eval()
adversarial_images = fgsm_attack.attack(model, images, labels)

# Evaluate original vs adversarial accuracy on the demonstration batch
with torch.no_grad():
    # Original predictions
    original_outputs = model(images)
    original_preds = torch.argmax(original_outputs, dim=1)
    original_accuracy = (original_preds == labels).float().mean()
    
    # Adversarial predictions
    adv_outputs = model(adversarial_images)
    adv_preds = torch.argmax(adv_outputs, dim=1)
    adv_accuracy = (adv_preds == labels).float().mean()

print(f"\n=== Attack Results (Single Batch Demo) ===")
print(f"Original accuracy: {original_accuracy:.4f}")
print(f"Adversarial accuracy: {adv_accuracy:.4f}")
print(f"Attack success rate: {1 - adv_accuracy:.4f}")

# Show perturbation statistics
perturbation = adversarial_images - images
print(f"\n=== Perturbation Statistics ===")
print(f"Max perturbation: {perturbation.abs().max():.6f}")
print(f"Mean perturbation: {perturbation.abs().mean():.6f}")
print(f"L2 norm of perturbation: {torch.norm(perturbation.flatten(), p=2):.6f}")

Attack Success Rate: 0.9894
Attack completed!
Generated adversarial images: 29688 batches

🔄 Demonstrating manual attack on one batch for comparison...

=== Attack Results (Single Batch Demo) ===
Original accuracy: 0.0000
Adversarial accuracy: 0.0000
Attack success rate: 1.0000

=== Perturbation Statistics ===
Max perturbation: 1.000000
Mean perturbation: 0.256131
L2 norm of perturbation: 117.291893


## 5. Training Example

### Basic Training Setup with Cross-Domain Evaluation

This section demonstrates how to set up training with our HuggingFace datasets. We'll show a basic training loop concept and how the cross-domain setup (CIFAR-10 train → CIFAR-10-C test) enables robustness evaluation.

In [30]:
# Configure training and create data loaders for cross-domain training
training_config = TrainConfig(
    epochs=2,  # Short for demo
    learning_rate=0.001,
    optimizer="adam",
    scheduler="step",
    save_checkpoint=True,
    checkpoint_interval=1,
    processor=str(device)
)

print("Training configuration created!")
print(f"Epochs: {training_config.epochs}")
print(f"Learning rate: {training_config.learning_rate}")
print(f"Optimizer: {training_config.optimizer}")

# Create data loaders for cross-domain training scenario
train_loader = DataLoaderFactory.create_dataloader(
    dataset=split_config_dataset['train'],  # Clean CIFAR-10 for training
    batch_size=32,
    shuffle=True
)

test_loader = DataLoaderFactory.create_dataloader(
    dataset=split_config_dataset['test'],   # CIFAR-10-C for robustness testing
    batch_size=32,
    shuffle=False
)

print(f"Train loader: {len(train_loader.dataset)} samples (clean CIFAR-10)")
print(f"Test loader: {len(test_loader.dataset)} samples (corrupted CIFAR-10-C)")

Training configuration created!
Epochs: 2
Learning rate: 0.001
Optimizer: adam
Train loader: 50000 samples (clean CIFAR-10)
Test loader: 950000 samples (corrupted CIFAR-10-C)


In [32]:
# Initialize training configuration - Standard Training (NOT adversarial training)
training_config = TrainConfig(
    model=model,
    train_loader=train_loader,  # Use train_loader, not test_loader
    epochs=2,
    learning_rate=0.001,
    optimizer="adam",
    criterion="cross_entropy",
    processor=device_config.processor,
    save_checkpoint=False,
    save_final_model=True,
    save_model_path="./models",
    save_model_name="huggingface_trained_model.pth"
)

print("Training configuration created successfully")
print(f"Training epochs: {training_config.epochs}")
print(f"Learning rate: {training_config.learning_rate}")
print(f"Optimizer: {training_config.optimizer}")
print(f"Device: {training_config.processor}")
print(f"Save model: {training_config.save_final_model}")

# Train the model
print("\nStarting model training...")
trainer = Trainer(training_config)
trainer.train()
print("Training completed successfully!")

Training configuration created successfully
Training epochs: 2
Learning rate: 0.001
Optimizer: adam
Device: mps
Save model: True

Starting model training...


  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1 - Average loss: 0.7763


 50%|█████     | 1/2 [00:59<00:59, 59.34s/it]

Epoch 2 - Average loss: 0.6407


100%|██████████| 2/2 [01:58<00:00, 59.47s/it]

Saved model to ./models/huggingface_trained_model.pth
Training completed successfully!
Training completed successfully!
